In [ ]:
#implement convnet from scratch
#Step !:
import numpy as np


In [ ]:
class Conv2D:
  def __init__(self, num_filters, filter_size):
    self.num_filters = num_filters
    self.filter_size = filter_size
    self.filters = np.random.randn(num_filters, filter_size, filter_size) / (filter_size*filter_size)

  def interate_regions(self, image):
    h, w = image.shape
    for i in range(h-self.filter_size+1):
      for j in range(w-self.filter_size+1):
        yield_image[i:i+self.filter_size, j:j+self.filter_size],i,j

  def forward(self, input):
    self.last_input = input
    h,w=input.shape
    output = np.zeros((h-self.filter_size+1, w-self.filter_size+1,  self.num_filters))
    for region, i, j in self.interate_regions(input):
      output[i,j] = np.sum(region*self.filters, axis=(1,2))
    return output

  def backward(self, d_L_d_out, learning_rate):
    d_L_d_filters = np.zeros(self.filters.shape)
    for region, i, j in self.interate_regions(self.last_input):
      for f in range(self.num_filters):
        d_L_d_filters[f]+=d_L_d_out[i,j,f]*region
    self.filters -= learning_rate*d_L_d_filters
    return None

In [ ]:
#STEP 3: ReLU activation function
class ReLU:
  def forward(self, input):
    self.last_input = input
    return np.maximum(0, input)
  def backward(self, d_L_d_out):
    return d_L_d_out*(self.last_input>0)

In [ ]:
#Stepo 4: Max pooling
class MaxPool2D:
  def iterate_regions(self, image):
    h, w, num_filters = image.shape
    for i in range(h//2):
      for j in range(w//2):
        yield_image[i*2:i*2+2, j*2:j*2+2],i,j
  def forward(self, input):
    self.last_input = input
    h, w, num_filters = input.shape
    output = np.zeros((h//2, w//2, num_filters))

    for region,i,j in self.iterate_regions(input):
      output[i,j] = np.amax(region, axis=(0,1))

    return output
  def backward(self, d_L_d_out):
    d_L_d_input = np.zeros(self.last_input.shape)

    for regions, i, j in self.iterate_regions(self.last_input):
      h, w, f = regions.shape
      amax = np.amax(regions, axis=(0,1))

      for i2 in range(h):
        for j2 in range(w):
          for f2 in range(f):
            if regions[i2, j2, f2] == amax[f2]:
              d_L_d_input[i*2+i2, j*2+j2,f2]= d_L_d_out[i,j,f2]
      return d_L_d_input